# YOLO-World Pseudo-Label + Selection

Yeni pipeline: YOLO-World pseudo-label, stratified filtering, embedding tabanlı çeşitlilik ve opsiyonel DINOv2/proxy-model adımları. Hücreler sıralı; GPU Colab A100 varsayımıyla hazırlandı.


In [ ]:
# Root of the shared project folder on Google Drive.
# Team members mounted it at different places in their Drives, so all paths
# below are built from this single constant. Change it to match your setup.
DRIVE_ROOT = "/content/drive/MyDrive/furniture-retrieval"

In [ ]:
# 1) Drive mount ve yol ayarları
from google.colab import drive
from pathlib import Path
import os

DRIVE_ROOT = Path(DRIVE_ROOT)
IMAGES_ARCHIVE = DRIVE_ROOT / 'all_images.tar.zst'
EMB_PARQUET = DRIVE_ROOT / 'embeddings' / 'all_embeddings.parquet'  # mevcut CLIP embeddings
OUTPUT_ROOT = DRIVE_ROOT / 'pseudo_label_outputs'
RAW_DET_JSONL = OUTPUT_ROOT / 'detections.jsonl'
SUMMARY_PARQUET = OUTPUT_ROOT / 'detections_summary.parquet'
FILTERED_PARQUET = OUTPUT_ROOT / 'filtered_manifest.parquet'
SELECTED_PARQUET = OUTPUT_ROOT / 'selected_manifest.parquet'
CACHE_DIR = Path('/content/cache_images')

print('Mounting Drive...')
drive.mount('/content/drive')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')
print(f'Archive: {IMAGES_ARCHIVE.exists()} | Embeddings: {EMB_PARQUET.exists()}')


In [ ]:
# 2) Bağımlılık kurulumu (A100 varsayımı: cu118)
%pip install -q ultralytics timm faiss-gpu pyarrow zstandard tqdm pandas numpy pillow

import torch
print('CUDA:', torch.cuda.is_available(), 'Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


In [ ]:
# 3) Arşiv inceleme (tar.zst içeriğini tam açmadan say)
import tarfile
import zstandard as zstd
from collections import Counter

assert IMAGES_ARCHIVE.exists(), 'Arşiv bulunamadı'

folder_counts = Counter()
with open(IMAGES_ARCHIVE, 'rb') as fh:
    dctx = zstd.ZstdDecompressor(max_window_size=2**31)
    with dctx.stream_reader(fh) as reader:
        with tarfile.open(fileobj=reader, mode='r|') as tar:
            for member in tar:
                if not member.isfile():
                    continue
                parts = Path(member.name).parts
                if len(parts) >= 2:
                    folder_counts[parts[0]] += 1

print(f'Klasor sayisi: {len(folder_counts)}')
for k, v in folder_counts.most_common(10):
    print(k, v)



In [ ]:
# 4) Lazy extraction helper'ları
import io
import hashlib
from PIL import Image

MAX_CACHE_FILES = 20000

def cache_key(path_str: str) -> Path:
    h = hashlib.md5(path_str.encode()).hexdigest()
    return CACHE_DIR / h


def extract_to_cache(member_name: str) -> Path:
    """Tar.zst icinden bir dosyayi gecici diske yazar ve yolu dondurur."""
    cached = cache_key(member_name)
    if cached.exists():
        return cached
    with open(IMAGES_ARCHIVE, 'rb') as fh:
        dctx = zstd.ZstdDecompressor(max_window_size=2**31)
        with dctx.stream_reader(fh) as reader:
            with tarfile.open(fileobj=reader, mode='r|') as tar:
                for member in tar:
                    if member.name == member_name:
                        f = tar.extractfile(member)
                        if f is None:
                            raise IOError('Dosya okunamadi')
                        data = f.read()
                        cached.write_bytes(data)
                        break
    return cached


def open_image_from_archive(member_name: str) -> Image.Image:
    path = extract_to_cache(member_name)
    return Image.open(path).convert('RGB')


def evict_cache(max_files: int = MAX_CACHE_FILES):
    files = sorted(CACHE_DIR.glob('*'), key=lambda p: p.stat().st_mtime)
    if len(files) <= max_files:
        return
    for p in files[: len(files) - max_files]:
        try:
            p.unlink()
        except FileNotFoundError:
            pass

print('Cache helperlar yuklendi')


In [ ]:
# 5) YOLO-World model yukleme ve sinif listesi
from ultralytics import YOLOWorld

# TODO: 42 sinif listesi buraya girilecek
YOLO_CLASSES = [
    # 'chair', 'sofa', ...
]

CONF_THRES = 0.1
IOU_THRES = 0.6
IMGSZ = 1024
BATCH_SIZE = 16

assert len(YOLO_CLASSES) > 0, 'YOLO sinif listesi doldurulmadi'

model = YOLOWorld('yolov8l-world.pt')
model.set_classes(YOLO_CLASSES)
model.to('cuda' if torch.cuda.is_available() else 'cpu')
print('Model hazir, sinif sayisi:', len(YOLO_CLASSES))


In [ ]:
# 6) YOLO-World inference ve JSONL kayit
import json
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm
import numpy as np

CHUNK_WRITE = 5000


def iter_archive_files():
    with open(IMAGES_ARCHIVE, 'rb') as fh:
        dctx = zstd.ZstdDecompressor(max_window_size=2**31)
        with dctx.stream_reader(fh) as reader:
            with tarfile.open(fileobj=reader, mode='r|') as tar:
                for member in tar:
                    if member.isfile():
                        yield member.name


def run_yolo_inference():
    RAW_DET_JSONL.unlink(missing_ok=True)
    SUMMARY_PARQUET.unlink(missing_ok=True)
    det_fp = open(RAW_DET_JSONL, 'a', encoding='utf-8')
    writer = None
    summary_rows = []

    batch_names = []
    batch_paths = []
    for name in tqdm(iter_archive_files(), desc='Infer'): 
        batch_names.append(name)
        batch_paths.append(str(extract_to_cache(name)))
        if len(batch_names) >= BATCH_SIZE:
            writer = process_batch(batch_names, batch_paths, det_fp, summary_rows, writer)
            batch_names, batch_paths = [], []
            evict_cache()
    if batch_names:
        writer = process_batch(batch_names, batch_paths, det_fp, summary_rows, writer)

    if summary_rows:
        table = pa.Table.from_pylist(summary_rows)
        if writer is None:
            writer = pq.ParquetWriter(SUMMARY_PARQUET, table.schema)
        writer.write_table(table)
        summary_rows.clear()
    if writer:
        writer.close()
    det_fp.close()
    print('Inference bitti')


def process_batch(names, paths, det_fp, summary_rows, writer):
    results = model.predict(paths, conf=CONF_THRES, iou=IOU_THRES, imgsz=IMGSZ, batch=BATCH_SIZE, verbose=False)
    for name, res in zip(names, results):
        boxes = res.boxes
        dets = []
        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            clss = boxes.cls.cpu().numpy().astype(int)
            areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
            hw = res.orig_shape[0] * res.orig_shape[1]
            for (x1, y1, x2, y2), c, cl, a in zip(xyxy, confs, clss, areas):
                dets.append({
                    'bbox': [float(x1), float(y1), float(x2), float(y2)],
                    'conf': float(c),
                    'cls': int(cl),
                    'cls_name': YOLO_CLASSES[int(cl)] if int(cl) < len(YOLO_CLASSES) else 'unk',
                    'area_rel': float(a / max(hw, 1))
                })
            max_conf = float(confs.max())
            cls_set = list(sorted(set(int(c) for c in clss)))
            area_rel = float(areas.sum() / max(hw, 1))
        else:
            max_conf = 0.0
            cls_set = []
            area_rel = 0.0

        det_fp.write(json.dumps({'image': name, 'detections': dets}) + '\n')
        summary_rows.append({
            'image': name,
            'has_box': len(dets) > 0,
            'max_conf': max_conf,
            'classes': cls_set,
            'area_rel': area_rel
        })

        if len(summary_rows) >= CHUNK_WRITE:
            table = pa.Table.from_pylist(summary_rows)
            if writer is None:
                writer = pq.ParquetWriter(SUMMARY_PARQUET, table.schema)
            writer.write_table(table)
            summary_rows.clear()
    return writer


# run_yolo_inference()  # uzun sure


In [ ]:
# 7) Hard filter ve histogram
import pandas as pd

def load_summary():
    return pq.read_table(SUMMARY_PARQUET).to_pandas()


def apply_hard_filters(max_conf_bg=0.25, bg_keep_frac=0.07, rare_thresh=2000, common_thresh=50000):
    df = load_summary()
    df['is_background'] = (~df['has_box']) | (df['max_conf'] < max_conf_bg)
    bg = df[df['is_background']]
    bg_keep = bg.sample(frac=bg_keep_frac, random_state=42) if len(bg) else bg

    # class histogram
    exploded = df.explode('classes')
    class_counts = exploded['classes'].value_counts()
    rare = set(class_counts[class_counts < rare_thresh].index.tolist())
    common = set(class_counts[class_counts > common_thresh].index.tolist())

    def is_rare(row):
        return any(cls in rare for cls in (row['classes'] or []))

    def is_common(row):
        return any(cls in common for cls in (row['classes'] or []))

    df['is_rare'] = df.apply(is_rare, axis=1)
    df['is_common'] = df.apply(is_common, axis=1)

    keep_mask = df['is_rare'] | (~df['is_background'])
    filtered = df[keep_mask]
    filtered = pd.concat([filtered, bg_keep]).drop_duplicates('image')
    filtered.to_parquet(FILTERED_PARQUET, index=False)

    print('Toplam:', len(df), 'Filtreden sonra:', len(filtered))
    print('Rare cls sayisi:', len(rare), '| Common cls sayisi:', len(common))
    return filtered, rare, common

# filtered_df, rare_cls, common_cls = apply_hard_filters()



In [ ]:
# 8) CLIP embeddinglerini yukle ve esle

clip_df = None
clip_map = None

def load_clip_embeddings(max_rows=None):
    global clip_df, clip_map
    clip_df = pd.read_parquet(EMB_PARQUET)
    if max_rows:
        clip_df = clip_df.head(max_rows)
    clip_df['image_basename'] = clip_df['image_name'].apply(lambda x: Path(x).name)
    clip_map = {row['image_basename']: row['embedding'] for _, row in clip_df.iterrows()}
    print('Embeddings yuklendi:', len(clip_map))


def attach_embeddings(df):
    assert clip_map is not None, 'Once load_clip_embeddings cagir'
    df = df.copy()
    df['clip_embedding'] = df['image'].apply(lambda p: clip_map.get(Path(p).name))
    return df

# load_clip_embeddings()



In [ ]:
# 9) Cesitlilik secimi (k-means + centroid yakinlari)
import faiss

def kmeans_select(emb_matrix, n_clusters=256, per_cluster=1, seed=42):
    emb_matrix = emb_matrix.astype('float32')
    kmeans = faiss.Kmeans(d=emb_matrix.shape[1], k=n_clusters, niter=20, verbose=False, seed=seed)
    kmeans.train(emb_matrix)
    D, I = faiss.search(emb_matrix, kmeans.centroids.shape[0])
    chosen = set()
    for row in I:
        for idx in row[:per_cluster]:
            chosen.add(int(idx))
    return list(chosen)


def diversity_sampling(filtered_df, common_cls, n_clusters=256, per_cluster=2):
    df = attach_embeddings(filtered_df)
    picked = []
    for cls_id in common_cls:
        subset = df[df['classes'].apply(lambda lst: cls_id in (lst or []))]
        subset = subset.dropna(subset=['clip_embedding'])
        if len(subset) < n_clusters:
            picked.append(subset)
            continue
        mat = np.vstack(subset['clip_embedding'].tolist()).astype('float32')
        idxs = kmeans_select(mat, n_clusters=n_clusters, per_cluster=per_cluster)
        picked.append(subset.iloc[idxs])
        print(f'Class {cls_id}: {len(subset)} -> {len(idxs)}')
    if picked:
        div_df = pd.concat(picked).drop_duplicates('image')
    else:
        div_df = pd.DataFrame()
    return div_df

# div_df = diversity_sampling(filtered_df, common_cls)



In [ ]:
# 10) DINOv2 adimi kaldirildi (kullanilmiyor)
# Bu adim simdilik devre disi birakildi. Ekstra ozellik gereksinimi olursa yeniden eklenebilir.



In [ ]:
# 11) Opsiyonel proxy YOLO (nano) egitimi ve belirsizlik skoru
from ultralytics import YOLO


def train_proxy_yolo(manifest_df, sample_size=50000, epochs=15):
    sample = manifest_df.sample(n=min(sample_size, len(manifest_df)), random_state=42)
    # Burada YOLO formatina donusum ve dataset hazirlama gerekiyor.
    # Var sayim: sample icin pseudo bbox'lar RAW_DET_JSONL'den cekilip label txt'ye yazilacak.
    # Kullanici gerektiginde dolduracak.
    print('TODO: proxy egitim icin label txt hazirla ve data.yaml olustur')


def score_uncertainty(proxy_model_path, images_list, conf_low=0.4, conf_high=0.7):
    model = YOLO(proxy_model_path)
    rows = []
    for img in tqdm(images_list, desc='Proxy infer'):
        res = model.predict(img, imgsz=IMGSZ, conf=0.05, verbose=False)[0]
        boxes = res.boxes
        if boxes is None or len(boxes) == 0:
            continue
        probs = boxes.probs if boxes.probs is not None else None
        if probs is None:
            continue
        entropies = -probs * torch.log(probs + 1e-9)
        ent = entropies.sum(dim=1)
        confs = boxes.conf
        mask = (confs >= conf_low) & (confs <= conf_high)
        if mask.any():
            rows.append({'image': img, 'entropy_mean': float(ent[mask].mean().cpu()), 'conf_mean': float(confs[mask].mean().cpu())})
    return pd.DataFrame(rows)

# proxy_scores = score_uncertainty('runs/detect/proxy/weights/best.pt', list_of_paths)



In [ ]:
# 12) Final manifest ve YOLOv11 hazirlik
import json


def load_raw_dets():
    with open(RAW_DET_JSONL, 'r') as f:
        for line in f:
            yield json.loads(line)


def build_final_manifest(filtered_df, diversity_df=None, proxy_scores=None, keep_entropy_top=200000):
    frames = [filtered_df]
    if diversity_df is not None:
        frames.append(diversity_df)
    if proxy_scores is not None:
        top = proxy_scores.sort_values('entropy_mean', ascending=False).head(keep_entropy_top)
        frames.append(top)
    final = pd.concat(frames).drop_duplicates('image')
    final.to_parquet(SELECTED_PARQUET, index=False)
    print('Final manifest satir:', len(final))
    return final


def write_yolo_labels(manifest_df, labels_dir: Path):
    labels_dir.mkdir(parents=True, exist_ok=True)
    det_map = {row['image']: row['detections'] for row in load_raw_dets()}
    for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc='Labels'):
        name = row['image']
        dets = det_map.get(name, [])
        if not dets:
            continue
        parts = Path(name).parts
        fname = '_'.join(parts) + '.txt'
        label_path = labels_dir / fname
        with open(label_path, 'w') as f:
            for d in dets:
                x1, y1, x2, y2 = d['bbox']
                cls_id = d['cls']
                # YOLO format (normalized xywh) icin en-boy gerekecek; burada bbox zaten piksel.
                # Kullanici resim boyutunu bilerek normalize etmeli veya result.orig_shape'i JSONL'e ekleyip kullanmali.
                f.write(f"{cls_id} {x1} {y1} {x2} {y2}\n")
    print('Label yazimi bitti')


def export_data_yaml(images_root: Path, labels_root: Path):
    yaml_text = f"""
path: {images_root}
train: images
val: images
names: {YOLO_CLASSES}
"""
    (labels_root / 'data.yaml').write_text(yaml_text)
    print('data.yaml hazir')

# final_df = build_final_manifest(filtered_df, div_df, proxy_scores)
# write_yolo_labels(final_df, OUTPUT_ROOT / 'labels')
# export_data_yaml(DRIVE_ROOT / 'all_images', OUTPUT_ROOT / 'labels')



In [ ]:
# 13) Secilen imagelari CSV'ye kaydet (folder, image_name, image_path)

def export_selected_csv(csv_path=OUTPUT_ROOT / 'selected_images.csv'):
    assert SELECTED_PARQUET.exists(), 'Once final manifest olustur (SELECTED_PARQUET)'
    df = pd.read_parquet(SELECTED_PARQUET)
    df['folder'] = df['image'].apply(lambda p: Path(p).parts[0] if len(Path(p).parts) > 0 else '')
    df['image_name'] = df['image'].apply(lambda p: Path(p).name)
    cols = ['folder', 'image_name', 'image']
    if 'classes' in df.columns:
        cols.append('classes')
    out_df = df[cols]
    out_df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path} | rows: {len(out_df)}')

export_selected_csv()  # Calistirdiginda Drive'a yazar


In [ ]:
# 14) Tam pipeline (tek hucreden calistir)

def run_full_pipeline(
    do_inference: bool = True,
    do_diversity: bool = True,
    n_clusters: int = 256,
    per_cluster: int = 2,
    export_csv: bool = True,
):
    assert len(YOLO_CLASSES) > 0, 'YOLO_CLASSES bos, sinif listesini doldur'

    if do_inference or not SUMMARY_PARQUET.exists():
        print('--- YOLO-World inference basliyor ---')
        run_yolo_inference()
    else:
        print('Inference atlandi, mevcut SUMMARY_PARQUET kullaniliyor')

    print('--- Hard filtre ---')
    filtered_df, rare_cls, common_cls = apply_hard_filters()
    print('Rare:', len(rare_cls), 'Common:', len(common_cls))

    print('--- CLIP embeddings yukleniyor ---')
    load_clip_embeddings()

    div_df = None
    if do_diversity and len(common_cls) > 0:
        print('--- Cesitlilik secimi (k-means) ---')
        div_df = diversity_sampling(filtered_df, common_cls, n_clusters=n_clusters, per_cluster=per_cluster)
        print('Diversity rows:', len(div_df))
    else:
        print('Cesitlilik adimi atlandi')

    print('--- Final manifest ---')
    final_df = build_final_manifest(filtered_df, div_df)

    if export_csv:
        print('--- CSV export ---')
        export_selected_csv()

    print('Tamamlandi. Satir sayisi:', len(final_df))
    return final_df

run_full_pipeline()  # tek satir calistir


## Calistirma Sirasi
1. Mount + path (Cell 1)
2. Dep kurulum (Cell 2)
3. Arşiv sayimi (Cell 3)
4. Cache helper (Cell 4)
5. YOLO sinif listesini doldur (Cell 5)
6. Inference (Cell 6 -> `run_yolo_inference()`)
7. Filtre/histogram (Cell 7 -> `apply_hard_filters()`)
8. Embedding yükle (Cell 8 -> `load_clip_embeddings()`)
9. Cesitlilik (Cell 9 -> `diversity_sampling(...)`)
10. DINOv2 adimi: simdilik kullanilmiyor
11. Opsiyonel proxy (Cell 11)
12. Final manifest + label export (Cell 12)
13. CSV export (Cell 13)
14. Tek-satir pipeline (Cell 14 -> `run_full_pipeline(...)`)

Not: YOLO label yazarken bbox normalizasyonu icin resim boyutunu JSONL'e eklemek istersen `process_batch` icinde `orig_shape` ekleyebilirsin.
